# Fra ZIP-høydedata til STL-modell

En komplett arbeidsflyt for **DTM GeoTIFF-fliser** fra Høydedata.no/Kartverket. Les rådata, sjekk den faktiske oppløsningen og den virkelige utstrekningen, velg et utsnitt, forhåndsvis terrenget og eksporter en **vanntett STL-modell**.

**Forberedelse:** Legg den originale høyde-ZIP-filen i `input/` (ikke pakk den ut). Start med `uv sync` og `uv run jupyter lab` i prosjektmappen. Kjør cellene ovenfra og ned.

STL-filen bruker **millimeter**, mens terreng-/kartdata bruker **meter**. *Høydeforskjell* = maksimal minus minimal terrenghøyde.

## 1. Finn ZIP-filen og importer biblioteker

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

candidate_files = sorted(Path('input').glob('*.zip'))
ZIP_PATH = candidate_files[0] if candidate_files else None
if ZIP_PATH is None:
    raise FileNotFoundError('Legg Høydedata ZIP-filen i input/ og kjør cellen igjen.')
print(f'Datakilde: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.1f} MB)')

## 2. Rask validering av RAW-data

Dette leser GeoTIFF-er direkte fra ZIP og viser original celleoppløsning, geografisk bredde/lengde og høyde i meter for hver datasamling. For å finne høyder leses hver flis én gang. Den totale utstrekningen kan inneholde hull; det er en **omsluttende boks**, ikke et løfte om kontinuerlig dekning.

In [ ]:
from terrain import scan_archive, summarize, crop_dem, export_stl

tiles = scan_archive(ZIP_PATH, read_heights=True)
source_names = sorted({r['source'] for r in tiles})
summaries = {s: summarize(tiles, source=s) for s in source_names}
summary_table = pd.DataFrame([{
    'Datasamling': s,
    'Fliser': v['tiles'],
    'CRS': v['crs'],
    'Rå oppløsning (m)': str(v['raw_resolution_m']),
    'Bredde (m)': v['bounding_width_m'],
    'Lengde (m)': v['bounding_length_m'],
    'Laveste høyde (m.o.h.)': v['min_height_m'],
    'Høyeste høyde (m.o.h.)': v['max_height_m'],
    'Høydeforskjell (m)': v['vertical_range_m'],
    'BBox (xmin, ymin, xmax, ymax)': v['bbox_m'],
} for s, v in summaries.items()])
display(summary_table)
print('NB! Bredden og lengden er en bounding box over flisene. Enkelte deler kan mangle datadekning.')

## 3. Velg datakilde og utsnitt

**Viktig:** ZIP-en din inneholder overlappende datasamlinger fra ulike år. Velg én. `BBOX` er UTM-koordinater i meter i samme CRS som filene. Eksemplet under er fra den opplastede filen; bytt koordinater for å lage et annet terrengutsnitt.

In [ ]:
# === BRUKERINNSTILLINGER: DATA ===
SOURCE = 'NDH Vågan-Hadsel sør 2pkt 2017'  # se datasamlingene vist over
BBOX = (467000.0, 7561600.0, 472400.0, 7566000.0) # Satt manuelt
N50_GDB_PATH = Path('n50_input/Basisdata_25833_N50Kartdata_FGDB.gdb')
ADD_LAKES = True  # Show the N50 vector water overlay and export a separate water STL.
OUTPUT_RESOLUTION_M = 5.0  # test med 2.0; bruk 0.5 for original gridoppløsning
FILL_HOLES = True         # unngå å finne på terreng hvor data mangler
MAX_CELLS = 8_000_000

assert SOURCE in summaries, f'SOURCE må være en av {source_names}'
if not N50_GDB_PATH.is_dir():
    raise FileNotFoundError(f'N50 File Geodatabase not found: {N50_GDB_PATH}')
print('Valgt utsnitt:', BBOX)
print('Virkelig bredde:', BBOX[2]-BBOX[0], 'm')
print('Virkelig lengde:', BBOX[3]-BBOX[1], 'm')
print('Valgt datakilde:',SOURCE)

## 3b. Preview N50 map data and water

The N50 data is vector map data. First preview every N50 feature in the crop, then the sea and lake polygons alone. The water mask is created later on the exact terrain grid.

In [ ]:
import importlib
import n50
importlib.reload(n50)  # Pick up changes when this notebook is rerun in an existing kernel.
from n50 import list_layers, read_features, water_areas, water_summary, water_mask, plot_features, plot_water, water_overlay_mesh

n50_layers = list_layers(N50_GDB_PATH)
display(n50_layers)
n50_features = read_features(N50_GDB_PATH, BBOX)
water = water_areas(n50_features)
display(water_summary(water))

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
plot_features(n50_features, BBOX, axes[0], title='All N50 features in selected bbox')
plot_water(water, BBOX, axes[1], title='N50 sea and lake polygons')
plt.tight_layout()
plt.show()

## 4. Lag utsnittet og sjekk høyder

En sammensatt høydemodell dannes bare for utsnittet. `OUTPUT_RESOLUTION_M` kan settes til råoppløsning 0,5 m for å bevare den opprinnelige rutenettheten. Grovere verdier brukes til raskere arbeid. Data som mangler, utløser feil som standard.

In [ ]:
dem = crop_dem(ZIP_PATH, BBOX, resolution_m=OUTPUT_RESOLUTION_M,
               source=SOURCE, max_cells=MAX_CELLS, fill_holes=FILL_HOLES)
h=dem['elevation_m']
n50_water_mask = water_mask(water, dem)
print(f'N50 water coverage: {n50_water_mask.mean():.2%} of terrain grid')
print(f"Kildens oppløsning: {dem['raw_resolution_m']} m")
print(f"Utrasterets oppløsning: {dem['output_resolution_m']} m")
print(f"Raster: {h.shape[1]} kolonner × {h.shape[0]} rader")
print(f"Dekningsgrad i utsnittet: {dem['coverage']:.2%}")
print(f"Faktisk areal (bounding box): {(dem['bbox_actual_m'][2]-dem['bbox_actual_m'][0]):.1f} m × {(dem['bbox_actual_m'][3]-dem['bbox_actual_m'][1]):.1f} m")
print(f"Min høyde: {h.min():.2f} m.o.h.")
print(f"Maks høyde: {h.max():.2f} m.o.h.")
print(f"Virkelig høydeforskjell: {np.ptp(h):.2f} m")

## 5b. Rask 3D-forhåndsvisning (nedsamplet)

In [ ]:
import plotly.graph_objects as go

# This is decimated only for a quick, interactive preview; STL export is unchanged.
preview_step = max(1, int(np.ceil(max(h.shape) / 180)))
preview = h[::preview_step, ::preview_step]
# Span the full crop extent: this aligns the decimated surface with N50 vector coordinates.
width_m = h.shape[1] * OUTPUT_RESOLUTION_M
length_m = h.shape[0] * OUTPUT_RESOLUTION_M
preview_x = np.linspace(width_m, 0, preview.shape[1])
preview_y = np.linspace(0, length_m, preview.shape[0])

# The separate N50-vector water mesh is exported below. This decimated terrain preview remains uncoloured.
def preview_surface_colors(heights_m, water_mask, color_water=False):
    if not color_water:
        return heights_m, 'Earth', float(heights_m.min()), float(heights_m.max())
    land = ~water_mask
    if not land.any():
        return np.zeros_like(heights_m), [[0, '#2166ac'], [1, '#2166ac']], 0, 1
    land_min, land_max = heights_m[land].min(), heights_m[land].max()
    land_scale = np.maximum(land_max - land_min, 1e-6)
    colors = np.where(land, 0.2 + 0.8 * (heights_m - land_min) / land_scale, 0.0)
    colorscale = [[0.0, '#2166ac'], [0.1999, '#2166ac'], [0.2, '#c2b280'],
                  [0.55, '#6b8e23'], [1.0, '#f5f5dc']]
    return colors, colorscale, 0, 1

preview_colors, preview_colorscale, preview_cmin, preview_cmax = preview_surface_colors(
    preview, np.zeros_like(preview, dtype=bool), False
)
interactive_fig = go.Figure(go.Surface(
    x=preview_x, y=preview_y, z=preview - preview.min(),
    surfacecolor=preview_colors, colorscale=preview_colorscale, cmin=preview_cmin, cmax=preview_cmax,
    colorbar=dict(title='Høyde/farge')
))
if ADD_LAKES:
    # N50 polygon mesh: its coast/lake edge retains vector detail in this 3D preview.
    WATER_PREVIEW_RAISE_M = 1.0  # Visual separation only; the DTM is not modified.
    water_preview_mesh, _ = water_overlay_mesh(
        water, dem, model_width_mm=width_m, vertical_exaggeration=1.0,
        base_mm=0.0, thickness_mm=0.05, raise_mm=WATER_PREVIEW_RAISE_M,
    )
    vertices = water_preview_mesh.vertices
    interactive_fig.add_trace(go.Mesh3d(
        x=width_m - vertices[:, 0], y=length_m - vertices[:, 1],
        z=vertices[:, 2] + h.min() - preview.min(),
        i=water_preview_mesh.faces[:, 0], j=water_preview_mesh.faces[:, 1], k=water_preview_mesh.faces[:, 2],
        color='#1677c8', opacity=1.0, name='N50 sea and lakes', showscale=False, flatshading=False,
    ))
interactive_fig.update_layout(
    title='Interaktiv 3D-forhåndsvisning',
    scene=dict(xaxis_title='Øst-vest (m)', yaxis_title='Nord-sør (m)', zaxis_title='Relieff (m)', aspectmode='data'),
    height=650, margin=dict(l=0, r=0, b=0, t=45),
)
interactive_fig.show()

## 6. Innstillinger for STL-modellen

`VERTICAL_EXAGGERATION = 1.0` betyr samme vertikale og horisontale målestokk. `2.0` dobler høydeforskjellene i den ferdige modellen. Minimumshøyden i utsnittet legges på toppen av en flat bunn på `BASE_MM`.

In [ ]:
# === BRUKERINNSTILLINGER: FYSISK MODELL ===
MODEL_WIDTH_MM = 180.0
VERTICAL_EXAGGERATION = 1.0
BASE_MM = 3.0
MAX_FACES = 10_000_000
OUTPUT_PATH = Path('output/lofoten_terrain.stl')
WATER_LAYER_THICKNESS_MM = 0.4  # Separate colour-printable water layer
WATER_RAISE_MM = WATER_LAYER_THICKNESS_MM  # Raised blue layer; reliable for a shared AMS print
WATER_OUTPUT_PATH = Path('output/lofoten_water_overlay.stl')

estimated_width_m = h.shape[1] * OUTPUT_RESOLUTION_M
estimated_length_m = h.shape[0] * OUTPUT_RESOLUTION_M
mm_per_m = MODEL_WIDTH_MM / estimated_width_m
print(f'Modellbredde: {MODEL_WIDTH_MM:.1f} mm')
print(f'Modellengde: {estimated_length_m * mm_per_m:.1f} mm')
print(f'Terrengrelieff i modellen: {np.ptp(h) * mm_per_m * VERTICAL_EXAGGERATION:.1f} mm')
print(f'Total modellhøyde inkl. bunn: {BASE_MM + np.ptp(h) * mm_per_m * VERTICAL_EXAGGERATION:.1f} mm')
print(f'Horisontal målestokk: omtrent 1:{1000/mm_per_m:.0f}')

## 7. Eksporter STL

STL er et triangulert volum med bunn og vegger. Eksporten kan bli stor ved 0,5 m. Hvis modellen blir for stor, øk `OUTPUT_RESOLUTION_M` eller reduser utsnittet.

In [ ]:
info = export_stl(dem, OUTPUT_PATH,
    model_width_mm=MODEL_WIDTH_MM,
    vertical_exaggeration=VERTICAL_EXAGGERATION,
    base_mm=BASE_MM,
    max_faces=MAX_FACES)
display(pd.DataFrame([info]).T.rename(columns={0:'Verdi'}))
print(f"\nSTL lagret: {OUTPUT_PATH.resolve()}")
print('Vanntett modell:',info['watertight'])

## 7b. Export N50 water overlay for a second colour

This is a separate, watertight STL. Its outline follows the original N50 vector polygons, while its height is sampled from the terrain. Import it with the terrain STL in a multi-material slicer and assign a second filament. STL files themselves do not retain colour.

In [ ]:
if ADD_LAKES:
    water_mesh, water_info = water_overlay_mesh(
        water, dem, model_width_mm=MODEL_WIDTH_MM,
        vertical_exaggeration=VERTICAL_EXAGGERATION, base_mm=BASE_MM,
        thickness_mm=WATER_LAYER_THICKNESS_MM, raise_mm=WATER_RAISE_MM,
    )
    WATER_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    water_mesh.export(WATER_OUTPUT_PATH, file_type='stl')
    water_info['path'] = str(WATER_OUTPUT_PATH)
    display(pd.DataFrame([water_info]).T.rename(columns={0: 'Water overlay'}))
else:
    print('ADD_LAKES is False: no water-overlay STL exported.')

## 8. Neste steg

Åpne STL-filen i Blender eller slicer. For mer detaljer, behold samme `BBOX`, men prøv `OUTPUT_RESOLUTION_M = 0.5` og øk `MAX_CELLS`/`MAX_FACES` ved behov. Husk at selve nøyaktigheten ikke blir bedre ved oppsampling utover rådata. Til utskrift kan det være nyttig å senke vertikal overdrivelse dersom modellen blir unormalt høy.